# Live demo — Outcome 4: Cross-chemistry limit (R² = −23.03)

**Run this cell-by-cell in front of the panel.** It downloads the *exact, unmodified*
validation script and data from this dissertation's own public code repository and
re-runs it from scratch — nothing here is pre-computed or faked.

Repository: https://github.com/Usharani1699/battery-degradation-analytics

What it does: trains an XGBoost SoH model on NASA + CALCE (LCO) only, with every
cell battery-grouped (GroupKFold — no cycle from one battery ever appears in both
train and test), then scores it against Oxford (NMC), which is held out completely.
This is the **R² = −23.03** figure quoted in the dissertation's Abstract and in
Appendix A.4 ("Validation correction record"), alongside its in-distribution CALCE
reference of **R² = 0.768**.

> **Note:** the dissertation also reports a second, separately audited external test
> (Random Forest trained on 7,106 NASA + BLAST + Severson rows, R² = −0.455 on all
> 72 withheld Oxford cycles). That test's BLAST/Severson feature files aren't in this
> lightweight public demo repo yet — its number is quoted directly from Appendix A.3/A.4
> of the dissertation. Ask me live and I can add those files and a matching notebook cell
> in a few minutes if you want it reproduced on screen too.

In [ ]:
# 1) Install the packages the script needs
!pip -q install xgboost scikit-learn pandas numpy

In [ ]:
# 2) Pull the real script + its input files straight from the public repo
import os, urllib.request

BASE = "https://raw.githubusercontent.com/Usharani1699/battery-degradation-analytics/main"

os.makedirs("data", exist_ok=True)
os.makedirs("04_Code/validation", exist_ok=True)
os.makedirs("04_Code/results", exist_ok=True)

files = {
    "data/Linked_Lab_Fleet_Degradation.csv":            f"{BASE}/data/Linked_Lab_Fleet_Degradation.csv",
    "04_Code/validation/cross_dataset_validation_CORRECTED.py": f"{BASE}/04_Code/validation/cross_dataset_validation_CORRECTED.py",
}
for local, url in files.items():
    urllib.request.urlretrieve(url, local)
    print(f"downloaded: {local}  ({os.path.getsize(local):,} bytes)")

In [ ]:
# 3) Run the dissertation's own corrected validation script, unedited, live
!python 04_Code/validation/cross_dataset_validation_CORRECTED.py

### Reading the output above

- **"CALCE only, battery-GROUPED ... GroupKFold"** block → this is the
  in-distribution reference, **R² = 0.768**: how well the model does on data from
  the same chemistry (LCO) it was trained on, with proper battery-level grouping.
- **"Oxford NMC — TRUE held-out cross-chemistry transfer"** block → this is the
  headline number, **R² = −23.03**: the same model applied, without any refitting,
  to a chemistry (NMC) it has never seen. A strongly negative R² here means the
  model is far worse than simply predicting the Oxford set's mean SoH — the
  clearest possible evidence that a relationship learned on one chemistry does not
  transfer to another without local calibration.
- This directly supersedes an earlier, leakage-affected figure of R² = −0.57 that
  appeared in draft materials — that number is not used anywhere in the final
  dissertation.